In [3]:
%pip install gudhi --quiet
%pip install scikit-tda --quiet
%pip install seaborn


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import ripser
import persim
from persim import PersistenceImager, PersLandscapeApprox, PersLandscapeExact
from persim.landscapes import plot_landscape_simple

In [5]:
LHR1 = pd.read_csv("LargeHypoxicRegion.csv")
LHR2 = pd.read_csv("LargeHypoxicRegion2.csv")

In [6]:
LHR1.head(5)

,x,y,PanCK+,CAIX+,Pimo+,Necrosis,Celltype
0,2.86,526.43,0,0,0,0,CD8
1,2.59,573.32,0,0,0,0,CD8
2,9.26,596.48,0,0,0,0,CD8
3,2.55,842.17,0,0,0,0,CD8
4,7.02,1044.55,0,0,0,0,CD8


In [7]:
LHR2.head(5)

,x,y,PanCK+,CAIX+,Pimo+,Necrosis,Celltype
0,2.59,67.20,0,0,0,0,CD8
1,7.99,106.15,0,1,0,0,CD8
2,3.25,373.03,0,0,0,0,CD8
3,8.40,480.41,0,0,0,0,CD8
4,3.93,572.57,0,0,0,0,CD8


In our dataset, we observe three types of immune cells:

- **CD8**: Their primary function is to identify and directly destroy infected or malignant cells. In the tumor microenvironment, a high spatial infiltration of CD8+ T cells within the tumor core is generally associated with a strong anti-tumor immune response and a better patient prognosis

- **CD68**: CD68 is a widely used pan-macrophage marker. These cells are specialized in phagocytosis (engulfing cellular debris and pathogens).

- **FoxP3**: Their primary role is to suppress immune responses and prevent autoimmunity. However, in cancer, a high infiltration of FoxP3 cells is generally a negative sign, as they actively inhibit the tumor-killing functions of the CD8 cells

In [ ]:
sns.set_theme(style="whitegrid")

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Distribution of cell types in the tumor region', fontsize=16, fontweight='bold')

sns.countplot(data=LHR1, x='Celltype', hue='Celltype', palette='viridis', legend=False, ax=axes[0])
axes[0].set_title('LHR1', fontsize=14)
axes[0].set_xlabel('Cell type', fontsize=12)
axes[0].set_ylabel('Total number of cells', fontsize=12)

sns.countplot(data=LHR2, x='Celltype', hue='Celltype', palette='viridis', legend=False, ax=axes[1])
axes[1].set_title('LHR2', fontsize=14)
axes[1].set_xlabel('Cell type', fontsize=12)
axes[1].set_ylabel('Total number of cells', fontsize=12)

plt.tight_layout()

We observe a similar distribution across both regions: CD8 and CD68 cells are present in roughly equal numbers, while FoxP3 cells are approximately half as numerous. This suggests a balanced presence of cytotoxic (CD8) and phagocytic (CD68) immune cells in the tumor microenvironment, with a smaller but notable regulatory (FoxP3) population. The relatively low proportion of FoxP3 cells compared to CD8 cells could indicate that the anti-tumor immune response is not entirely suppressed in these regions.

Beyond cell type composition, a key factor shaping the tumor microenvironment is oxygen availability. We now investigate the hypoxic state of each cell using molecular markers present in our dataset.

**Hypoxia (Low Oxygen)**: Hypoxia refers to regions within the tissue that are deprived of adequate oxygen. As solid tumors grow rapidly, they often outstrip their local blood supply, creating these oxygen-starved areas. In cancer, hypoxia is a strong negative prognostic factor: it drives tumor aggressiveness, increases resistance to therapies (like radiation), and creates a highly immunosuppressive microenvironment that inhibits the function of CD8+ cells while promoting suppressive cells like FoxP3+.

**CAIX+ / Pimo+ (Hypoxia Markers)**: These are the specific markers used in the dataset to identify hypoxic cells. CAIX (Carbonic Anhydrase IX) is an endogenous protein produced by cells to survive low oxygen levels, while Pimo (Pimonidazole) is an exogenous chemical tracer that specifically binds to oxygen-deprived cells. In the following visualization, we use the CAIX+ marker to assess the hypoxic state of each cell type.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('State of hypoxia (CAIX+) according to cell type', fontsize=16, fontweight='bold')

sns.countplot(data=LHR1, x='Celltype', hue='CAIX+', palette='Set2', ax=axes[0])
axes[0].set_title('LHR1', fontsize=14)
axes[0].set_xlabel('Cell type', fontsize=12)
axes[0].set_ylabel('Total number of cells', fontsize=12)
axes[0].legend(title='Hypoxia marker (CAIX+)', labels=['Normoxia (0)', 'Hypoxia (1)'])

sns.countplot(data=LHR2, x='Celltype', hue='CAIX+', palette='Set2', ax=axes[1])
axes[1].set_title('LHR2', fontsize=14)
axes[1].set_xlabel('Cell type', fontsize=12)
axes[1].set_ylabel('Total number of cells', fontsize=12)
axes[1].legend(title='Hypoxia marker (CAIX+)', labels=['Normoxia (0)', 'Hypoxia (1)'])

plt.tight_layout()

### Observations

**CD8 and CD68 — predominantly normoxic:** In both LHR1 and LHR2, the vast majority of CD8 (cytotoxic) and CD68 (macrophage) cells are in a normoxic state, with very few cells expressing the CAIX+ marker. This indicates that these immune populations reside mainly in well-oxygenated areas of the tumor.

**FoxP3 — a strong hypoxic signature:** In contrast, the FoxP3 (regulatory T-cell) population exhibits a significant proportion of hypoxic cells. This pattern is consistent across both regions, suggesting it is not an artifact but a genuine biological feature.

**Interpretation:** The co-localization of FoxP3 cells with hypoxic zones is consistent with the known biology of the tumor microenvironment: hypoxia promotes immunosuppression by recruiting and sustaining regulatory T-cells, while excluding cytotoxic CD8 cells. This reinforces the idea that oxygen-deprived regions act as immune-privileged niches where the tumor can evade the anti-tumor response.

In [ ]:
df1 = LHR1.copy()
df2 = LHR2.copy()

for df in [df1, df2]:
    conditions = [
        (df['Necrosis'] == 1),
        (df['PanCK+'] == 1)
    ]
    choices = ['Necrotic Tissue', 'Tumor Cells (PanCK+)']
    df['Tissue_Structure'] = np.select(conditions, choices, default='Stroma / Immune')

fig, axes = plt.subplots(2, 3, figsize=(24, 18))
fig.suptitle('Spatial Distribution of Cells within the Tumor Microenvironment', fontsize=22, fontweight='bold')

for row, (df, label) in enumerate([(df1, 'LHR1'), (df2, 'LHR2')]):
    sns.scatterplot(
        data=df, x='x', y='y', hue='Celltype',
        palette={'CD8': '#3b528b', 'CD68': '#21918c', 'FoxP3': '#5ec962'},
        s=15, alpha=0.8, edgecolor=None, ax=axes[row, 0]
    )
    axes[row, 0].set_title(f'1. Immune Infiltration (Cell Types) — {label}', fontsize=16)
    axes[row, 0].set_xlabel('X coordinate (µm)', fontsize=12)
    axes[row, 0].set_ylabel('Y coordinate (µm)', fontsize=12)
    axes[row, 0].legend(title='Cell Type', markerscale=2)

    df_sorted_hypoxia = df.sort_values(by='CAIX+')
    sns.scatterplot(
        data=df_sorted_hypoxia, x='x', y='y', hue='CAIX+',
        palette={0: '#e0e0e0', 1: '#d62728'},
        s=15, alpha=0.8, edgecolor=None, ax=axes[row, 1]
    )
    axes[row, 1].set_title(f'2. Hypoxic Regions (CAIX+) — {label}', fontsize=16)
    axes[row, 1].set_xlabel('X coordinate (µm)', fontsize=12)
    axes[row, 1].set_ylabel('')
    axes[row, 1].legend(title='Hypoxia Status', labels=['Normoxia (0)', 'Hypoxia (1)'], markerscale=2)

    df_sorted_structure = df.sort_values(by='Tissue_Structure', ascending=False)
    sns.scatterplot(
        data=df_sorted_structure, x='x', y='y', hue='Tissue_Structure',
        palette={'Stroma / Immune': '#e0e0e0', 'Tumor Cells (PanCK+)': '#ff7f0e', 'Necrotic Tissue': '#000000'},
        s=15, alpha=0.8, edgecolor=None, ax=axes[row, 2]
    )
    axes[row, 2].set_title(f'3. Tumor Architecture & Necrosis — {label}', fontsize=16)
    axes[row, 2].set_xlabel('X coordinate (µm)', fontsize=12)
    axes[row, 2].set_ylabel('')
    axes[row, 2].legend(title='Tissue Component', markerscale=2)

plt.tight_layout(rect=[0, 0, 1, 0.96])

### Observations

**Immune infiltration (panel 1):** CD8, CD68 and FoxP3 cells are spread across the tissue but tend to concentrate at the periphery of the tumor mass rather than within its core. This pattern of immune exclusion is a hallmark of aggressive solid tumors.

**Hypoxic regions (panel 2):** The CAIX+ hypoxic cells (red) form well-defined clusters that are not randomly distributed. They are primarily located around the necrotic cores, forming a gradient of decreasing oxygen from the tumor periphery inward.

**Tumor architecture (panel 3):** The necrotic tissue (black) sits at the center of the tumor mass (orange/PanCK+), surrounded by the hypoxic belt observed in panel 2. The stroma and immune cells (grey) populate the outer margins.

**Cross-panel interpretation:** These three views reveal a concentric spatial organization: a necrotic core, surrounded by a hypoxic ring rich in FoxP3 cells, and an oxygenated periphery where CD8 and CD68 cells are more prevalent. This architecture shows how the tumor's physical structure and metabolic environment shape immune cell distribution, effectively creating a barrier that shields the tumor interior from immune attack. This pattern is consistent across both LHR1 and LHR2.

## Topological Data Analysis (TDA)

The previous visualizations gave us a qualitative understanding of the spatial organization. We now turn to **Topological Data Analysis** to quantify these spatial patterns rigorously.

The idea is to study the **shape** of the point clouds formed by each cell type using persistent homology. For each cell type and region, we build a filtration of simplicial complexes (Vietoris-Rips) at increasing distance scales and track the appearance and disappearance of topological features:

- **$H_0$ (connected components):** Points start as isolated components that merge as the scale increases. Features that persist over a long range indicate well-separated clusters, while short-lived features reflect dense, uniformly distributed cells.
- **$H_1$ (loops/holes):** These capture circular voids in the point cloud. A persistent loop suggests a ring-like arrangement of cells surrounding an empty region — potentially a hole in the immune infiltration pattern (e.g., around a necrotic or hypoxic zone).

In [ ]:
datasets = {
    'CD8': {'LHR1': LHR1, 'LHR2': LHR2},
    'CD68': {'LHR1': LHR1, 'LHR2': LHR2},
    'FoxP3': {'LHR1': LHR1, 'LHR2': LHR2},
}

all_diagrams = {}
for celltype in ['CD8', 'CD68', 'FoxP3']:
    for label, df in [('LHR1', LHR1), ('LHR2', LHR2)]:
        data = df[df['Celltype'] == celltype][['x', 'y']].values
        np.random.shuffle(data)
        data = data[:1500]
        print(f"Calculating {celltype} {label}: {len(data)} points...")
        result = ripser.ripser(data, maxdim=1)
        all_diagrams[(celltype, label)] = result['dgms']

# Store in named variables for downstream cells
diagrams_CD8_1 = all_diagrams[('CD8', 'LHR1')]
diagrams_CD68_1 = all_diagrams[('CD68', 'LHR1')]
diagrams_Fox_1 = all_diagrams[('FoxP3', 'LHR1')]
diagrams_CD8_2 = all_diagrams[('CD8', 'LHR2')]
diagrams_CD68_2 = all_diagrams[('CD68', 'LHR2')]
diagrams_Fox_2 = all_diagrams[('FoxP3', 'LHR2')]

fig, axes = plt.subplots(2, 3, figsize=(24, 16))
fig.suptitle('Persistence Diagrams', fontsize=20, fontweight='bold')

celltypes = ['CD8', 'CD68', 'FoxP3']
labels = ['LHR1', 'LHR2']

for row, label in enumerate(labels):
    for col, celltype in enumerate(celltypes):
        ax = axes[row, col]
        persim.plot_diagrams(all_diagrams[(celltype, label)], show=False,
                             title=f"{celltype} — {label}",
                             labels=['$H_0$', '$H_1$'], ax=ax)
        ax.set_xlabel("Birth", fontsize=12)
        ax.set_ylabel("Death", fontsize=12)
        ax.grid(True, linestyle='--', alpha=0.6)

plt.tight_layout(rect=[0, 0, 1, 0.96])

### Observations

**$H_0$ — connectivity:** Across all cell types and both regions, the $H_0$ points cluster near the diagonal, meaning connected components merge quickly as the scale grows. This confirms that the immune cells form a single, densely connected spatial distribution rather than isolated clusters.

**$H_1$ — loops in CD8 and CD68:** Both CD8 and CD68 diagrams exhibit numerous $H_1$ features, many with moderate to high persistence (far from the diagonal). This indicates the presence of significant circular voids in their spatial distribution — consistent with the immune exclusion zones observed in the spatial maps (areas around necrotic/hypoxic cores where these cells are absent).

**$H_1$ — loops in FoxP3:** The FoxP3 diagrams show fewer and generally less persistent $H_1$ features. This is expected since FoxP3 cells, being co-localized with hypoxic zones, fill in some of the voids that CD8 and CD68 cells avoid. Their spatial distribution is more homogeneous, with fewer "holes".

In [ ]:
all_diagrams_list = [diagrams_CD8_1, diagrams_CD68_1, diagrams_Fox_1, diagrams_CD8_2, diagrams_CD68_2, diagrams_Fox_2]

all_births = [np.max(d[1][:, 0]) for d in all_diagrams_list if d[1].size > 0]
all_pers = [np.max(d[1][:, 1] - d[1][:, 0]) for d in all_diagrams_list if d[1].size > 0]
global_max_birth = max(all_births)
global_max_pers = max(all_pers)

pimgr = PersistenceImager(
    pixel_size=global_max_birth / 50,
    birth_range=(0, global_max_birth * 1.1),
    pers_range=(0, global_max_pers * 1.1),
)

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
fig.suptitle('Persistence Images', fontsize=20, fontweight='bold')

celltypes = ['CD8', 'CD68', 'FoxP3']
labels = ['LHR1', 'LHR2']

for row, label in enumerate(labels):
    for col, celltype in enumerate(celltypes):
        ax = axes[row, col]
        dgm = all_diagrams_list[row * 3 + col]
        img = pimgr.transform(dgm[1])
        pimgr.plot_image(img, ax=ax)
        ax.set_title(f"{celltype} — {label}", fontsize=14)

plt.tight_layout(rect=[0, 0, 1, 0.96])

### Vectorizations

To move from visual inspection to quantitative comparison, we transform the persistence diagrams into fixed-size vector representations:

- **Persistence Images:** Each diagram is converted into a 2D image by placing a Gaussian kernel on each point in the birth-persistence plane. This produces a stable, differentiable summary that can be used as input for machine learning models or statistical tests. Bright regions indicate where topological features concentrate.

- **Persistence Landscapes:** Each diagram is converted into a sequence of piecewise-linear functions (one per "depth level"). The first landscape captures the most persistent features, the second captures the next most persistent, etc. Landscapes live in a Banach space, which allows computing means, norms, and statistical tests directly.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(24, 14))
fig.suptitle('Persistence Landscapes', fontsize=20, fontweight='bold')

celltypes = ['CD8', 'CD68', 'FoxP3']
labels = ['LHR1', 'LHR2']
diagrams_map = {
    ('CD8', 'LHR1'): diagrams_CD8_1, ('CD68', 'LHR1'): diagrams_CD68_1, ('FoxP3', 'LHR1'): diagrams_Fox_1,
    ('CD8', 'LHR2'): diagrams_CD8_2, ('CD68', 'LHR2'): diagrams_CD68_2, ('FoxP3', 'LHR2'): diagrams_Fox_2,
}

for row, label in enumerate(labels):
    for col, celltype in enumerate(celltypes):
        ax = axes[row, col]
        pla = PersLandscapeApprox(dgms=diagrams_map[(celltype, label)], hom_deg=1)
        plot_landscape_simple(pla, depth_range=range(10), ax=ax)
        ax.set_title(f"{celltype} — {label}", fontsize=14)

plt.tight_layout(rect=[0, 0, 1, 0.96])